# SEED-EXT — three more adapter seeds for arms A and B

**Why:** the adapter-clustered upper confidence bound is the one that generalises, and at n=3 it
uses t₀.₉₉₅,₂ = **9.92**. It is currently driven entirely by arm B's between-seed spread
(sd 4.19e-05 vs arm A's 7.6e-06), which inflates λ_U from 0.32% to 0.70%.

At n=6, t₀.₉₉₅,₅ = **4.03** — less than half — and the between-seed sd is itself estimated from
six points instead of three. This is the cheapest available improvement to the headline bound.

**What it does NOT fix:** the device level still has n=2 (A and B). Claims about *devices in
general* need the multi-device replication (Dresden has six camera models with 5 instances each).
This fixes the *seed* level only — say so in the paper.

**Training data is byte-identical to the base run** (`BASE_ROOT/train_png/A_raw` and `B_raw`), so
only the seed differs. Separate `EXT_ROOT` because `TRAIN_SEEDS` is inside the hashed config —
changing it in the base notebook would invalidate all fourteen existing adapters.

Run order: B0 → B1 → B2 → B3 → B4. Roughly 5 h unattended.

## Run all and leave

1. **Add your HF token to Colab Secrets** (key icon, left sidebar) as `HF_TOKEN`, with
   *Notebook access* enabled. Without it the token cell prompts and the run blocks.
2. **Runtime → Change runtime type → A100** (bf16 is required; the preflight halts on fp16).
3. **Runtime → Run all.** Approve the Google Drive mount when it appears — that is the only
   interaction. After that it runs end to end.

Every stage is resumable and skip-if-complete, so if the session drops, just Run all again;
finished work is detected and skipped.

`STAGES` is already set to the full sequence. To run a single stage instead, edit that tuple.


In [ ]:
# %% TOKEN — RUN THIS FIRST. Non-interactive if HF_TOKEN is in Colab Secrets.
#   Colab: key icon in the left sidebar -> add secret named HF_TOKEN -> enable "Notebook access".
#   Without a secret this falls back to a prompt, which BLOCKS an unattended run.
import os, sys, subprocess
try:
    import huggingface_hub  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)

_tok = ""
try:
    from google.colab import userdata
    _tok = (userdata.get("HF_TOKEN") or "").strip()
    if _tok: print("[TOKEN] loaded from Colab Secrets")
except Exception:
    pass
if not _tok:
    _tok = os.environ.get("HF_TOKEN", "").strip()
if not _tok:
    import getpass
    print("[TOKEN] no Colab Secret found — prompting (this blocks an unattended run)")
    _tok = getpass.getpass("HF token (SD-3.5 license must be accepted on your account): ").strip()
if not _tok:
    raise SystemExit("No HF token. SD-3.5 is a gated repo and every training/generation "
                     "stage will fail without it.")
os.environ["HF_TOKEN"] = _tok
from huggingface_hub import login
login(token=_tok)
print("[TOKEN] Hugging Face login OK — safe to Run all and walk away")


In [ ]:
# %% B0 — config, env, base-run reuse
import os, sys, json, glob, math, time, gc, shutil, hashlib, subprocess, random
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
import csv as _csv

from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE = "/content/drive/MyDrive"
if not os.path.isdir(DRIVE): raise SystemExit("Drive not mounted.")
BASE_ROOT = os.path.join(DRIVE, "inv_channel/E_INV_P0_v3")
EXT_ROOT  = os.path.join(DRIVE, "inv_channel/E_SEEDEXT")

@dataclass
class CFG:
    STAGES: tuple = ("B0", "B1", "B2", "B3", "B4")
    HF_MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    CROP: int = 1024
    N_T: int = 50
    NEW_SEEDS: tuple = (3, 4, 5)          # base run used (0,1,2)
    ROLES_TRAIN: tuple = ("A", "B")       # only the two IUT arms need more seeds
    LORA_RANK: int = 16
    LORA_TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    G_PER_ADAPTER: int = 500
    GEN_SEED_BASE: int = 770000           # SAME bank as the base run -> poolable
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    N_PERM: int = 10000
    GRAD_CKPT: bool = False

C = CFG()
MEAS = C.CROP
CFG_SHA = hashlib.sha256(json.dumps(
    {k: v for k, v in asdict(C).items() if k != "STAGES"}, sort_keys=True, default=str).encode()).hexdigest()
for d in ["", "adapters", "gens", "csv", "figs"]:
    os.makedirs(os.path.join(EXT_ROOT, d), exist_ok=True)
CACHE = "/content/e_seed_cache"; os.makedirs(CACHE, exist_ok=True)

def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED — not in C.STAGES={C.STAGES}"); return False

ROLES = ("A", "B", "C", "D", "D2")
def all_arms(): return [(f"{r}_raw_s{s}_r16", r, s) for s in C.NEW_SEEDS for r in C.ROLES_TRAIN]
def all_tags(): return [t for t, _, _ in all_arms()]
def arm_of(t):  return next(a for a in all_arms() if a[0] == t)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets", "diffusers>=0.31",
                "transformers", "peft", "accelerate", "safetensors", "sentencepiece", "protobuf"],
               check=False)
import numpy as np, pywt, torch
import torch.nn.functional as tF
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_DTYPE = torch.bfloat16 if DEV == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
np.seterr(divide="raise", invalid="raise")
_MEASURE_DEV = DEV
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN not set — run the TOKEN cell at the top first.")

def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
def append_row(p, h, r):
    new = not os.path.exists(p)
    with open(p, "a") as f:
        if new: f.write(",".join(h) + "\n")
        f.write(",".join(str(x) for x in r) + "\n")
def done_keys(p, cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def atomic_json_dump(o, p, **kw):
    t = p + ".tmp"; json.dump(o, open(t, "w"), **kw); os.replace(t, p)

def base_train_dir(role): return os.path.join(BASE_ROOT, "train_png", f"{role}_raw")
def K_base(role): return np.load(os.path.join(BASE_ROOT, "fingerprints", f"K_{role}_E2.npy"))

for r in C.ROLES_TRAIN:
    n = len(glob.glob(os.path.join(base_train_dir(r), "*.png")))
    if n != C.N_T: HALT(f"base training PNGs for {r}_raw: {n}, expected {C.N_T}")
print(f"[B0] base training data OK — reusing byte-identical PNGs from {BASE_ROOT}")
print(f"[B0] new arms: {all_tags()}")
# --- peft/torchao compatibility -----------------------------------------------------
# peft's is_torchao_available() RAISES ImportError when torchao is older than its
# minimum (Colab ships 0.10.0, peft wants >=0.16.0) instead of returning False, which
# kills add_adapter(). Nothing here is quantised, so disable the dispatcher outright.
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
    try:
        import peft.tuners.lora.torchao as _plt
        _plt.is_torchao_available = lambda: False
    except Exception:
        pass
    print("[ENV] peft torchao dispatcher disabled")
except Exception as _e:
    print("[ENV] torchao patch skipped:", type(_e).__name__)

print(f"[B0] protocol SHA={CFG_SHA[:16]}  device={DEV} dtype={TRAIN_DTYPE}")

In [ ]:
# %% forensic core (fp32 measurement path, identical to the base run)
def _cp(fp): return os.path.join(CACHE, hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16] + ".npy")

def load_lum_crop(fp, use_cache=True):
    if use_cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im = im.convert("RGB"); W, H = im.size
        a = np.asarray(im.crop(((W-MEAS)//2, (H-MEAS)//2, (W-MEAS)//2+MEAS, (H-MEAS)//2+MEAS)), np.float32)
    lum = (0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).astype(np.float32)
    if use_cache: np.save(_cp(fp), lum)
    return lum

def _conv2(x, k):
    xt = torch.from_numpy(np.ascontiguousarray(x))[None, None]; kt = torch.from_numpy(k)[None, None]
    if _MEASURE_DEV == "cuda":
        xt, kt = xt.cuda(), kt.cuda()
        return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0, 0].cpu().numpy()
    return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0, 0].numpy()

def wavelet_residual(img):
    s0 = np.float32(2.0)**2
    co = pywt.wavedec2(img, "db8", level=4, mode="periodization")
    out = [co[0]*0.0]
    for (cH, cV, cD) in co[1:]:
        band = []
        for ch in (cH, cV, cD):
            ch = ch.astype(np.float32); vmin = None
            for w in (3, 5, 7, 9):
                v = np.maximum(_conv2(ch*ch, np.ones((w, w), np.float32)/(w*w)) - s0, 0.0)
                vmin = v if vmin is None else np.minimum(vmin, v)
            band.append(ch*(s0/(vmin + s0)))
        out.append(tuple(band))
    n = pywt.waverec2(out, "db8", mode="periodization")[:img.shape[0], :img.shape[1]]
    r = n - n.mean(); sd = float(r.std())
    if not np.isfinite(sd) or sd == 0.0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)

def _ncc_batch(a, B):
    a = a - a.mean(); na = a.norm()
    Bc = B - B.mean(dim=(1, 2), keepdim=True)
    nb = Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)

In [ ]:
# %% B1 — train the six new adapters
def adapter_dir(t): return os.path.join(EXT_ROOT, "adapters", t)
def _expected(t):
    _, role, seed = arm_of(t)
    return {"tag": t, "role": role, "seed": seed, "rank": C.LORA_RANK, "steps": C.STEPS,
            "meas": MEAS, "config_sha": CFG_SHA, "model": C.HF_MODEL, "dtype": str(TRAIN_DTYPE)}
def adapter_is_valid(t):
    mp = os.path.join(adapter_dir(t), "train_meta.json")
    if not os.path.exists(os.path.join(adapter_dir(t), "pytorch_lora_weights.safetensors")): return False
    m = json.load(open(mp)) if os.path.exists(mp) else None
    return m is not None and all(m.get(k) == v for k, v in _expected(t).items()) and m.get("effective", False)

def _prompt_embeds():
    p = os.path.join(EXT_ROOT, "adapters", "_prompt_embeds.pt")
    if os.path.exists(p):
        d = torch.load(p, map_location=DEV); return d["pe"].to(DEV), d["pp"].to(DEV)
    from diffusers import StableDiffusion3Pipeline
    pipe = StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True).to(DEV)
    with torch.no_grad():
        pe, _, pp, _ = pipe.encode_prompt(prompt=C.CAPTION, prompt_2=C.CAPTION, prompt_3=C.CAPTION,
                                          device=DEV, num_images_per_prompt=1,
                                          do_classifier_free_guidance=False)
    torch.save({"pe": pe.cpu(), "pp": pp.cpu()}, p)
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return pe.to(DEV), pp.to(DEV)

def train_arm(tag, pe, pp):
    _, role, seed = arm_of(tag)
    if adapter_is_valid(tag): print(f"[B1] skip valid {tag}"); return
    if DEV != "cuda": HALT("B1 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    out = adapter_dir(tag)
    if os.path.exists(out): shutil.move(out, out + f"__stale_{int(time.time())}")
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    try:
        pipe = StableDiffusion3Pipeline.from_pretrained(
            C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True,
            text_encoder=None, text_encoder_2=None, text_encoder_3=None,
            tokenizer=None, tokenizer_2=None, tokenizer_3=None).to(DEV)
    except Exception:
        pipe = StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True).to(DEV)
        pipe.text_encoder = pipe.text_encoder_2 = pipe.text_encoder_3 = None
    sch, vae, tr = pipe.scheduler, pipe.vae, pipe.transformer
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.LORA_RANK, lora_alpha=C.LORA_RANK,
                              init_lora_weights="gaussian", target_modules=list(C.LORA_TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    params = [p for p in tr.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=C.LR, weight_decay=1e-4)
    lrs = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=math.ceil(C.STEPS/C.GRAD_ACC))
    opt.zero_grad(set_to_none=True)
    files = sorted(glob.glob(os.path.join(base_train_dir(role), "*.png")))
    cache = torch.stack([torch.from_numpy(
        (np.asarray(Image.open(f).convert("RGB"), np.float32).copy()/127.5 - 1.0)).permute(2, 0, 1)
        for f in files])
    sf = float(vae.config.scaling_factor); sh = float(vae.config.shift_factor or 0.0)
    ts, sg = sch.timesteps.to(DEV), sch.sigmas.to(DEV)
    n_ts = int(sch.config.num_train_timesteps)
    if len(sg) != n_ts or len(ts) != n_ts: HALT("scheduler array length mismatch — pin diffusers")
    g = torch.Generator().manual_seed(seed); losses = []; t0 = time.time()
    print(f"\n[B1] {tag}: role={role} seed={seed}, {C.STEPS} steps")
    for st in range(C.STEPS):
        idx = torch.randint(0, cache.shape[0], (C.BATCH,), generator=g)
        px = cache[idx].to(DEV, dtype=TRAIN_DTYPE)
        with torch.no_grad():
            lat = vae.encode(px).latent_dist.sample(); lat = ((lat - sh)*sf).to(TRAIN_DTYPE)
        noise = torch.randn_like(lat)
        u = torch.sigmoid(torch.randn(lat.shape[0], device=DEV, dtype=torch.float32))
        ii = (u*n_ts).long().clamp_(0, n_ts - 1)
        sig = sg[ii].to(lat.dtype).view(-1, 1, 1, 1)
        pred = tr(hidden_states=((1.0 - sig)*lat + sig*noise).to(TRAIN_DTYPE), timestep=ts[ii],
                  encoder_hidden_states=pe.repeat(lat.shape[0], 1, 1).to(TRAIN_DTYPE),
                  pooled_projections=pp.repeat(lat.shape[0], 1).to(TRAIN_DTYPE),
                  return_dict=False)[0]
        loss = tF.mse_loss(pred.float(), (noise - lat).float())/C.GRAD_ACC
        loss.backward()
        if (st + 1) % C.GRAD_ACC == 0:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); lrs.step(); opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach())*C.GRAD_ACC)
        if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
        if (st + 1) % 400 == 0:
            el = time.time() - t0
            print(f"    {st+1}/{C.STEPS} loss={np.mean(losses[-400:]):.5f} "
                  f"eta={(el/(st+1))*(C.STEPS-st-1)/60:.1f}min")
    sd = get_peft_model_state_dict(tr)
    bnorm = float(sum(float(v.float().norm()) for k, v in sd.items() if "lora_B" in k))
    if not (bnorm > 1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
        HALT(f"{tag}: DEAD adapter (||lora_B||={bnorm:.3e})")
    os.makedirs(out, exist_ok=True)
    StableDiffusion3Pipeline.save_lora_weights(save_directory=out, transformer_lora_layers=sd)
    m = _expected(tag); m.update({"loss_tail": float(np.mean(losses[-100:])), "lora_B_norm": bnorm,
                                  "effective": True, "minutes": (time.time()-t0)/60})
    atomic_json_dump(m, os.path.join(out, "train_meta.json"), indent=2)
    print(f"[B1] saved {tag}: ||lora_B||={bnorm:.2f} tail={m['loss_tail']:.5f} ({m['minutes']:.1f} min)")
    del pipe, tr, vae, cache; gc.collect(); torch.cuda.empty_cache()

if stage("B1"):
    pe, pp = _prompt_embeds()
    for i, t in enumerate(all_tags(), 1):
        print(f"[B1] adapter {i}/{len(all_tags())}: {t}"); train_arm(t, pe, pp)
    bad = [t for t in all_tags() if not adapter_is_valid(t)]
    if bad: HALT(f"B1 incomplete: {bad}")
    print(f"[B1] PASS — {len(all_tags())} adapters")

In [ ]:
# %% B2 — generation (same paired seed bank as the base run, so results pool cleanly)
def gen_dir(t): return os.path.join(EXT_ROOT, "gens", t)
def bank_(): return [(C.CAPTION, C.GEN_SEED_BASE + j) for j in range(C.G_PER_ADAPTER)]

def b2_generate():
    if DEV != "cuda": HALT("B2 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    b = bank_()
    pipe = StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True); prev = False
    try:
        for tag in all_tags():
            d_ = gen_dir(tag); os.makedirs(d_, exist_ok=True)
            for _t in glob.glob(os.path.join(d_, "*.tmp")): os.remove(_t)
            miss = [i for i in range(C.G_PER_ADAPTER)
                    if not os.path.exists(os.path.join(d_, f"{i:05d}.png"))]
            if not miss: print(f"[B2] skip complete {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev = False
            pipe.load_lora_weights(adapter_dir(tag)); prev = True
            print(f"[B2] {tag}: {len(miss)} missing")
            for s in range(0, len(miss), C.GEN_BATCH):
                ix = miss[s:s+C.GEN_BATCH]
                gs = [torch.Generator(device=DEV).manual_seed(b[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs = pipe(prompt=[b[i][0] for i in ix], num_inference_steps=C.GEN_STEPS,
                                guidance_scale=C.CFG_SCALE, height=MEAS, width=MEAS, generator=gs).images
                for i, im in zip(ix, imgs):
                    t_ = os.path.join(d_, f"{i:05d}.png.tmp")
                    im.save(t_, format="PNG")   # PIL cannot infer format from .tmp
                    os.replace(t_, os.path.join(d_, f"{i:05d}.png"))
                if (s // C.GEN_BATCH) % 25 == 0: print(f"    {s+len(ix)}/{len(miss)}")
            print(f"[B2] complete {tag}")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[B2] PASS")

if stage("B2"): b2_generate()

In [ ]:
# %% B3 — measurement (same five fingerprints as the base run)
def b3_measure():
    K = torch.stack([torch.from_numpy(K_base(r)) for r in ROLES]).to(_MEASURE_DEV)
    hdr = ["tag", "gen_idx", "K", "rho_mult"]
    p = os.path.join(EXT_ROOT, "csv", "b3_measure.csv")
    done = done_keys(p, ["tag", "gen_idx", "K"])
    pool = ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            paths = [os.path.join(gen_dir(tag), f"{i:05d}.png") for i in range(C.G_PER_ADAPTER)]
            todo = [i for i in range(C.G_PER_ADAPTER)
                    if not all((tag, str(i), r) in done for r in ROLES)]
            if not todo: print(f"[B3] {tag}: complete"); continue
            fut = {i: pool.submit(load_lum_crop, paths[i], False) for i in todo[:6]}
            t0 = time.time()
            for pos, i in enumerate(todo):
                Z_np = fut.pop(i).result()
                nx = pos + 6
                if nx < len(todo): fut[todo[nx]] = pool.submit(load_lum_crop, paths[todo[nx]], False)
                W = torch.from_numpy(wavelet_residual(Z_np)).to(_MEASURE_DEV)
                Z = torch.from_numpy(Z_np).to(_MEASURE_DEV)
                rm = _ncc_batch(W, Z.unsqueeze(0)*K).cpu().numpy()
                for j, r in enumerate(ROLES):
                    if (tag, str(i), r) in done: continue
                    append_row(p, hdr, [tag, i, r, f"{rm[j]:.6e}"])
            print(f"[B3] {tag}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    finally:
        pool.shutdown(wait=True); gc.collect()
        if DEV == "cuda": torch.cuda.empty_cache()
    n = len(done_keys(p, ["tag", "gen_idx", "K"]))
    exp = len(all_tags())*C.G_PER_ADAPTER*len(ROLES)
    if n != exp: HALT(f"B3 incomplete: {n}/{exp}")
    print(f"[B3] PASS — {n}/{exp} rows")

if stage("B3"): b3_measure()

In [ ]:
# %% B4 — POOLED analysis over all six seeds per arm: the new adapter-clustered bound
def b4_pooled():
    import pandas as pd
    from scipy import stats as sps
    base = pd.read_csv(os.path.join(BASE_ROOT, "csv", "s5_measure.csv"))
    ext  = pd.read_csv(os.path.join(EXT_ROOT,  "csv", "b3_measure.csv"))
    s1   = pd.read_csv(os.path.join(BASE_ROOT, "csv", "s1_positive_control.csv"))

    def rho(df, tag, K):
        return df[(df.tag == tag) & (df.K == K)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    def real_contrast(role, other):
        a = s1[(s1.role_img == role) & (s1.role_K == role)].sort_values("idx")["rho_mult"].to_numpy(float)
        b = s1[(s1.role_img == role) & (s1.role_K == other)].sort_values("idx")["rho_mult"].to_numpy(float)
        n = min(len(a), len(b)); return a[:n] - b[:n]
    R_real = 0.5*(real_contrast("A", "B").mean() + real_contrast("B", "A").mean())

    ALPHA_SIM = 0.01
    z_b = sps.norm.ppf(1 - ALPHA_SIM/2)
    out = {"config_sha": CFG_SHA, "R_real": float(R_real), "arms": {}}
    print(f"R_real = {R_real:.5e}\n")
    for arm, ks, ko in (("A", "A", "B"), ("B", "B", "A")):
        seed_means, seed_ids = [], []
        for s in (0, 1, 2):
            t = f"{arm}_raw_s{s}_r16"
            seed_means.append((rho(base, t, ks) - rho(base, t, ko)).mean()); seed_ids.append(f"base_s{s}")
        for s in C.NEW_SEEDS:
            t = f"{arm}_raw_s{s}_r16"
            seed_means.append((rho(ext, t, ks) - rho(ext, t, ko)).mean()); seed_ids.append(f"ext_s{s}")
        m = np.array(seed_means); k = len(m)
        se = float(m.std(ddof=1)/np.sqrt(k)); tc = float(sps.t.ppf(1 - ALPHA_SIM/2, df=k-1))
        U = float(m.mean() + tc*se)
        out["arms"][arm] = {"seed_ids": seed_ids, "seed_means": m.tolist(), "theta": float(m.mean()),
                            "se_cluster": se, "n_adapters": k, "t_crit": tc, "U_cluster": U}
        print(f"arm {arm}: n={k} adapters")
        for i, v in zip(seed_ids, m): print(f"   {i:9s} {v:+.3e}")
        print(f"   theta={m.mean():+.3e}  sd={m.std(ddof=1):.3e}  t_crit={tc:.2f}  U={U:+.3e}\n")

    U_dev = max(out["arms"]["A"]["U_cluster"], out["arms"]["B"]["U_cluster"])
    theta_sym = 0.5*(out["arms"]["A"]["theta"] + out["arms"]["B"]["theta"])
    out["U_device"] = U_dev; out["theta_sym"] = theta_sym
    out["lambda_hat"] = theta_sym/R_real; out["lambda_U"] = U_dev/R_real
    print("="*70)
    print(f"POOLED (n={out['arms']['A']['n_adapters']} adapters/arm)")
    print(f"  theta_sym = {theta_sym:+.3e}   lambda_hat = {theta_sym/R_real:+.4%}")
    print(f"  lambda_U (adapter-clustered, 99% simultaneous) = {U_dev/R_real:.4%}")
    print(f"  previous value at n=3 was 0.6993%")
    print("="*70)
    atomic_json_dump(out, os.path.join(EXT_ROOT, "seedext_pooled.json"), indent=2)

if stage("B4"): b4_pooled()

## Running it

| Session | STAGES | Time | Check |
|---|---|---|---|
| 1 | `("B0",)` | seconds | base training PNGs found for A_raw and B_raw |
| 2 | `("B1",)` | ~2.5 h (6 adapters) | every ‖lora_B‖ non-zero; compare against the base run's 65.8–72.3 |
| 3 | `("B2",)` | ~2 h (3000 gens) | all tags complete |
| 4 | `("B3","B4")` | ~15 min | the new **λ_U** |

**What to expect.** t₀.₉₉₅ drops from 9.92 to 4.03. If the six seed means have a similar spread to
the current three, λ_U should land near **0.3%** rather than 0.70%. If arm B's spread persists at
n=6, that is itself informative — it says B's adapters genuinely disagree, and the honest bound
really is looser than arm A alone would suggest.

**Watch arm B specifically.** Its current three seed means are −3.42e-05, +1.36e-05, +4.92e-05 —
a spread five times arm A's. Three more seeds will tell you whether that was bad luck or a real
property of the noisier B fingerprint.

**Wording:** this fixes the *seed* level only. The device level is still n=2. Do not upgrade the
claim to "devices in general" on the strength of this — that needs the multi-device replication.